# UCL Data Transformation (Teams)

**Install and Imports** 

In [0]:
%pip install azure-storage-blob

In [0]:
import json
import io
import pandas as pd
from datetime import datetime
from azure.storage.blob import BlobServiceClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, to_timestamp, trim, when, lit, min as spark_min, max as spark_max, current_timestamp, upper, length
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, TimestampType, BooleanType

**Configuration**

In [0]:
STORAGE_ACCOUNT_NAME= "Your Storage account name" #Replace it with your azure storage account name
STORAGE_ACCOUNT_KEY= "Your Storage account key" #Replace it with your azure storage account key

#Azure Storage account name
BRONZE_CONTAINER = "bronze"
SILVER_CONTAINER = "silver"

conn_str = f"DefaultEndpointsProtocol=https;AccountName={STORAGE_ACCOUNT_NAME};AccountKey={STORAGE_ACCOUNT_KEY};EndpointSuffix=core.windows.net"
try:
    az_client = BlobServiceClient.from_connection_string(conn_str)
    print(f"Azure client connected: {STORAGE_ACCOUNT_NAME}")
    print(f"Spark version: {spark.version}")
except Exception as e:
    print("Connection failed: ", e)

**Helper Functions**

In [0]:
# Return list with all the blobs inside the container
def list_bronze_blobs(prefix: str) -> list:
    """Return list of blob names matching prefix in Bronze container."""
    container_client = az_client.get_container_client(BRONZE_CONTAINER)
    blobs = [
        b.name
        for b in container_client.list_blobs(name_starts_with=prefix)
    ]
    print(f"Found {len(blobs)} blobs with prefix '{prefix}':")
    for b in blobs:
        print(f"  {b}")
    return blobs

def download_bronze_json(blob_path: str) -> dict | list | None:
    """
    Download a JSON file from Bronze container.
    Returns parsed Python object or None if failed.
    """
    try:
        blob_client = az_client.get_blob_client(
            container = BRONZE_CONTAINER,
            blob      = blob_path,
        )
        raw_bytes = blob_client.download_blob().readall()
        data      = json.loads(raw_bytes)
        size_kb   = len(raw_bytes) / 1024
        print(f"Downloaded: {blob_path} ({size_kb:.1f} KB)")
        return data
    except Exception as e:
        print(f"Failed to download {blob_path}: {e}")
        return None

def upload_silver_parquet(df, blob_path: str) -> bool:
    """ Upload Spark DataFrame to Azure Silver as Parquet. """
    try:
        print(f"Collecting {df.count()} rows...")

        # Using collect method convert dataframe into python object and cast as Dict.
        rows_as_dicts = [row.asDict(recursive=True) for row in df.collect()]
        print(f"Converted {len(rows_as_dicts)} rows to plain dicts")

        # creating pandas dataframe by dict
        pandas_df = pd.DataFrame(rows_as_dicts)
        print(f"Pandas shape: {pandas_df.shape}")

        # Serialise to Parquet bytes in memory
        buffer = io.BytesIO()
        pandas_df.to_parquet(
            buffer,
            engine      = "pyarrow",
            index       = False,
            compression = "snappy",
        )
        buffer.seek(0)
        parquet_bytes = buffer.getvalue()
        size_kb = len(parquet_bytes) / 1024
        print(f"Parquet serialised: {size_kb:.1f} KB")

        # Upload to Azure Silver
        blob_client = az_client.get_blob_client(container = SILVER_CONTAINER, blob = blob_path,)
        blob_client.upload_blob(parquet_bytes, overwrite=True)
        print(f"Uploaded to Silver: {SILVER_CONTAINER}/{blob_path}")
        return True

    except MemoryError:
        print("MemoryError: DataFrame too large.")
        return False

    except Exception as e:
        print(f" Upload failed: {e}")
        return False

print("--Helper functions defined--")

**Read Bronze + Explore raw data**

In [0]:
bronze_blobs = list_bronze_blobs(prefix="teams/")

latest_blob = sorted(bronze_blobs)[-1]
print(f"\nUsing latest file: {latest_blob}")

raw_data = download_bronze_json(latest_blob)

if not raw_data:
    raise ValueError("Failed to load Bronze match data — check Azure connection")

teams_raw = raw_data.get("teams", [])
metadata    = raw_data.get("_ingestion_metadata", {})

print(f"\n--- Bronze File Summary ---")
print(f"Ingested at  : {metadata.get('ingested_at', 'unknown')}")
print(f"Source       : {metadata.get('source', 'unknown')}")
print(f"Match count  : {len(teams_raw)}")

# Show structure of first match to understand the schema
if teams_raw:
    print(f"\n--- Raw match keys (first record) ---")
    for key, val in teams_raw[0].items():
        print(f"  {key}: {type(val).__name__} = {str(val)[:70]}")

**PySpark Transformation (JSON -> DataFrame)**

In [0]:
# Method to flatten one record
def flatten_team(team: dict) -> dict:
    """
    Flatten one raw team record from the API response.
    Extracts nested area, competition, and running competitions fields.
    """
    # area = country/region the team is from
    area = team.get("area", {})

    # runningCompetitions = list of competitions team is in this season
    # We take the first one that matches UCL (code="CL") if present
    competitions = team.get("runningCompetitions", [])
    ucl_entry    = next(
        (c for c in competitions if c.get("code") == "CL"),
        competitions[0] if competitions else {},
    )
    return {
        # Team identifiers 
        "team_id"           : team.get("id"),
        "team_name"         : team.get("name"),
        "team_short_name"   : team.get("shortName"),
        "team_tla"          : team.get("tla"),       

        # Derived display name 
        "team_display_name" : team.get("shortName") or team.get("name"),

        # Club details
        "founded_year"      : team.get("founded"),
        "club_colours"      : team.get("clubColors"),
        "venue_name"        : team.get("venue"),
        "website"           : team.get("website"),

        # Area / Country 
        "area_id"           : area.get("id"),
        "area_name"         : area.get("name"),
        "area_code"         : area.get("code"),

        # Competition context 
        "competition_id"    : ucl_entry.get("id"),
        "competition_name"  : ucl_entry.get("name"),
        "competition_code"  : ucl_entry.get("code"),
    }


# ── Flatten all teams 
flattened = [flatten_team(t) for t in teams_raw]
print(f"Flattened {len(flattened)} team records")

df_bronze = spark.createDataFrame(flattened)

print("\n--- Bronze DataFrame Schema ---")
df_bronze.printSchema()

# ── Apply Silver transformations 
df_silver = (
    df_bronze

    # ── Cast numeric fields 
    .withColumn("team_id",       col("team_id").cast(IntegerType()))
    .withColumn("area_id",       col("area_id").cast(IntegerType()))
    .withColumn("competition_id",col("competition_id").cast(IntegerType()))
    .withColumn("founded_year",  col("founded_year").cast(IntegerType()))

    # ── Standardise string fields 
    .withColumn("team_name",         trim(col("team_name")))
    .withColumn("team_short_name",   trim(col("team_short_name")))
    .withColumn("team_tla",          upper(trim(col("team_tla"))))
    .withColumn("team_display_name", trim(col("team_display_name")))
    .withColumn("area_name",         trim(col("area_name")))
    .withColumn("area_code",         upper(trim(col("area_code"))))

    # ── Derive is_ucl_participant 
    # True if competition_code confirms UCL participation
    .withColumn(
        "is_ucl_participant",
         when(col("competition_code") == "CL", True)
        .otherwise(False)
        .cast(BooleanType())
    )

    # ── Silver metadata 
    .withColumn("silver_processed_at", current_timestamp().cast(StringType()))
    .withColumn("source_blob",         lit(latest_blob))

    # ── Final column selection 
    .select(
        "team_id",
        "team_name",
        "team_short_name",
        "team_tla",
        "team_display_name",
        "founded_year",
        "club_colours",
        "venue_name",
        "website",
        "area_id",
        "area_name",
        "area_code",
        "competition_id",
        "competition_name",
        "competition_code",
        "is_ucl_participant",
        "silver_processed_at",
        "source_blob",
    )
)

print("\n--- Silver DataFrame Schema ---")
df_silver.printSchema()

print("\n--- Sample (5 rows) ---")
df_silver.select(
    "team_id", "team_name", "team_tla",
    "area_name", "founded_year", "venue_name"
).show(5, truncate=False)

**Data Quality Checks**

In [0]:
print("Running data quality checks...")
print("=" * 60)

dq_passed  = True
dq_results = {}

# Check 1: Row count matches expected UCL team count 
# UCL group stage has exactly 32 teams
row_count  = df_silver.count()
check_name = "row_count_gte_32"
passed     = row_count >= 32
dq_results[check_name] = {"passed": passed, "value": row_count}
status = "PASS" if passed else "WARN"
print(f"{status} | {check_name} | rows={row_count}")
# Warning only — knockout rounds may have fewer teams in the file

#  Check 2: team_id has no nulls 
null_count = df_silver.filter(col("team_id").isNull()).count()
check_name = "team_id_not_null"
passed     = null_count == 0
dq_results[check_name] = {"passed": passed, "value": null_count}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | nulls={null_count}")
if not passed:
    dq_passed = False

# Check 3: team_id is unique 
# Critical — team_id is the primary key of this dimension table
total_rows   = df_silver.count()
distinct_ids = df_silver.select("team_id").distinct().count()
check_name   = "team_id_unique"
passed       = total_rows == distinct_ids
dq_results[check_name] = {
    "passed": passed,
    "value": f"{distinct_ids}/{total_rows} unique",
}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | distinct={distinct_ids} total={total_rows}")
if not passed:
    dq_passed = False

# Check 4: team_name has no nulls 
null_names = df_silver.filter(col("team_name").isNull()).count()
check_name = "team_name_not_null"
passed     = null_names == 0
dq_results[check_name] = {"passed": passed, "value": null_names}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | nulls={null_names}")
if not passed:
    dq_passed = False

# Check 5: all teams have area_name 
null_areas = df_silver.filter(col("area_name").isNull()).count()
check_name = "area_name_not_null"
passed     = null_areas == 0
dq_results[check_name] = {"passed": passed, "value": null_areas}
status = "PASS" if passed else "WARN"
print(f"{status} | {check_name} | nulls={null_areas}")
# Warning only — area may be missing for some clubs

# Check 6: TLA codes are 3 characters 
invalid_tla = (
    df_silver
    .filter(col("team_tla").isNotNull())
    .filter(length(col("team_tla")) != 3)
    .count()
)
check_name = "tla_length_3"
passed     = invalid_tla == 0
dq_results[check_name] = {"passed": passed, "value": invalid_tla}
status = "PASS" if passed else "WARN"
print(f"{status} | {check_name} | invalid_length={invalid_tla}")

#Final Verdict
print("=" * 60)
if dq_passed:
    print("ALL CRITICAL DATA QUALITY CHECKS PASSED")
else:
    failed = [k for k, v in dq_results.items() if not v["passed"]]
    print(f"CRITICAL DQ FAILED — {failed}")
    raise ValueError(f"Silver write blocked. Fix: {failed}")

### Write Silver Parquet to Azure

In [0]:
# Write Silver DataFrame to Azure ADLS Gen2 
today       = datetime.now().strftime("%Y-%m-%d")
silver_blob = f"teams/silver_ucl_teams_{today}.parquet"

print(f"Writing Silver to: {SILVER_CONTAINER}/{silver_blob}")
print(f"Row count: {df_silver.count()}")

success = upload_silver_parquet(df_silver, silver_blob)

if not success:
    raise RuntimeError("Silver teams write failed")

#  Verify 
blob_client  = az_client.get_blob_client(
    container = SILVER_CONTAINER,
    blob      = silver_blob,
)
size_kb = blob_client.get_blob_properties().size / 1024

print(f"\nSilver teams verified")
print(f"   Path    : {SILVER_CONTAINER}/{silver_blob}")
print(f"   Size    : {size_kb:.1f} KB")
print(f"   Rows    : {df_silver.count()}")
print(f"   Columns : {len(df_silver.columns)}")

# Final display 
print("\n--- UCL Teams Silver — final output ---")
(
    df_silver
    .select(
        "team_id", "team_display_name", "team_tla",
        "area_name", "founded_year", "venue_name",
    )
    .orderBy("team_name")
    .show(10, truncate=False)
)